# ACE inhibitor regressor — pChEMBL workflow (revised)

Quantitative counterpart to the IC50-cutoff classifier (notebook 1). Same compounds, same scaffold-grouped fold assignment, same fingerprints.

- Reads the cleaned tables written by notebook 1 (biochemical/binding assays only, prodrugs removed). Run notebook 1 first.
- **Only exact ("=") IC50 compounds are used.** Censored `> 10,000 nM` inactives have no measured pChEMBL and are ignored here (they are still part of the classifier).
- Fold assignment is computed on the full table, exactly as in notebook 1, then restricted to exact-value compounds, so both models see identical groupings.
- The external set is evaluated **once**, for the model already chosen by training-only CV; a mean-predictor baseline and Spearman correlation give context for R².
- Tanimoto k-NN/kernel live in `ace_tanimoto.py` (written to the project folder) so saved models can be reloaded anywhere.

In [1]:
from pathlib import Path
import os, sys, json, warnings
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path('/content/drive/MyDrive/ACE_Workflow')
except ImportError:
    ROOT = Path(os.environ.get('ACE_WORKFLOW_ROOT', Path.cwd())).resolve()
if not (ROOT / 'ace_config.py').exists():
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ace_config.py').exists()), ROOT)
if not (ROOT / 'ace_config.py').exists():
    raise FileNotFoundError('Could not find ace_config.py. Mount Drive or set ACE_WORKFLOW_ROOT.')
os.environ['ACE_WORKFLOW_ROOT'] = str(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import ace_config

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
RDLogger.DisableLog('rdApp.*')

import joblib, numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVR

RANDOM_STATE = ace_config.RANDOM_STATE
CPU_WORKERS = ace_config.CPU_WORKERS
N_SPLITS, FP_RADIUS, FP_SIZE = 5, 2, 2048   # must match notebook 1
PATHS, INPUTS = ace_config.PATHS, ace_config.INPUTS
OUT, PROCESSED, MODEL_DIR = PATHS['ML_REGRESSION'], PATHS['DATA_PROCESSED'], PATHS['ML_MODELS']
for d in (OUT, MODEL_DIR): Path(d).mkdir(parents=True, exist_ok=True)

# Importable helper module (so pickled models do not depend on this notebook's __main__).
(ROOT / 'ace_tanimoto.py').write_text('"""Tanimoto helpers shared by the ACE regression notebooks.\n\nKept in a real module (not the notebook\'s __main__) so joblib pickles of\nTanimotoKNN / SVR(kernel=tanimoto_kernel) can be loaded from any notebook.\n"""\nimport numpy as np\nfrom sklearn.base import BaseEstimator, RegressorMixin\n\n\ndef tanimoto_kernel(A, B):\n    """Tanimoto (Jaccard) similarity between rows of binary matrices A and B."""\n    A = np.asarray(A, dtype=np.float32)\n    B = np.asarray(B, dtype=np.float32)\n    dot = A @ B.T\n    denom = A.sum(axis=1)[:, None] + B.sum(axis=1)[None, :] - dot\n    return np.divide(dot, denom, out=np.zeros_like(dot), where=denom > 0)\n\n\ndef max_tanimoto_similarity(X, X_ref, chunk=2000):\n    """Max Tanimoto similarity of each row of X to any row of X_ref."""\n    X = np.asarray(X)\n    out = np.empty(len(X), dtype=np.float32)\n    for start in range(0, len(X), chunk):\n        out[start:start + chunk] = tanimoto_kernel(X[start:start + chunk], X_ref).max(axis=1)\n    return out\n\n\nclass TanimotoKNN(BaseEstimator, RegressorMixin):\n    """Similarity-weighted k-NN regressor using Tanimoto similarity."""\n\n    def __init__(self, n_neighbors=5):\n        self.n_neighbors = n_neighbors\n\n    def fit(self, X, y):\n        self.X_ = np.asarray(X, dtype=np.float32)\n        self.y_ = np.asarray(y, dtype=float)\n        return self\n\n    def predict(self, X, chunk=2000):\n        X = np.asarray(X, dtype=np.float32)\n        k = min(self.n_neighbors, len(self.y_))\n        pred = np.empty(len(X))\n        for start in range(0, len(X), chunk):\n            sim = tanimoto_kernel(X[start:start + chunk], self.X_)\n            idx = np.argpartition(-sim, k - 1, axis=1)[:, :k]\n            w = np.take_along_axis(sim, idx, axis=1)\n            y_nb = self.y_[idx]\n            wsum = w.sum(axis=1)\n            pred[start:start + chunk] = np.where(\n                wsum > 0, (w * y_nb).sum(axis=1) / np.where(wsum > 0, wsum, 1), self.y_.mean())\n        return pred\n\n\ndef predict_in_chunks(model, X, chunk=5000):\n    """Predict large libraries without building one giant kernel matrix."""\n    X = np.asarray(X)\n    return np.concatenate([model.predict(X[i:i + chunk]) for i in range(0, len(X), chunk)])\n', encoding='utf-8')
import importlib, ace_tanimoto; importlib.reload(ace_tanimoto)
from ace_tanimoto import TanimotoKNN, tanimoto_kernel, max_tanimoto_similarity, predict_in_chunks

fp_generator = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_SIZE)
def fingerprint_matrix(smiles):
    rows = []
    for s in smiles:
        mol = Chem.MolFromSmiles(s)
        if mol is None: raise ValueError(f'Invalid SMILES: {s}')
        arr = np.zeros(FP_SIZE, dtype=np.uint8)
        DataStructs.ConvertToNumpyArray(fp_generator.GetFingerprint(mol), arr)
        rows.append(arr)
    return np.vstack(rows)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Load the cleaned tables produced by notebook 1.
train_file = PROCESSED / 'chembl1808_ic50_clean.csv'
external_file = PROCESSED / 'chembl2096989_ic50_external_disjoint.csv'
for f in (train_file, external_file):
    if not f.exists(): raise FileNotFoundError(f'{f} not found - run notebook 1 first.')
train_all = pd.read_csv(train_file)
external_all = pd.read_csv(external_file)
need = {'canonical_smiles', 'pchembl', 'active_ic50', 'scaffold', 'ic50_censored'}
for name, df in (('train', train_all), ('external', external_all)):
    miss = need - set(df.columns)
    if miss: raise KeyError(f'{name}: missing columns {sorted(miss)} - rerun notebook 1 with the updated ace_config.py')
assert not set(train_all.canonical_smiles) & set(external_all.canonical_smiles), 'train/external overlap'

# Fingerprints and folds for the FULL table (identical to the classifier), regression uses exact rows only.
X_all = fingerprint_matrix(train_all.canonical_smiles)
y_cls_all = train_all.active_ic50.to_numpy()
groups_all = train_all.scaffold.astype(str).to_numpy()
exact_train = ~train_all.ic50_censored.astype(bool).to_numpy()
n_groups = len(np.unique(groups_all))
if n_groups < N_SPLITS: raise ValueError(f'Only {n_groups} scaffold groups - cannot run {N_SPLITS}-fold grouped CV.')
splits_all = list(StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(X_all, y_cls_all, groups_all))
splits = [(tr[exact_train[tr]], va[exact_train[va]]) for tr, va in splits_all]
for k, (tr, va) in enumerate(splits, 1):
    assert not set(groups_all[tr]) & set(groups_all[va]), f'scaffold leakage in fold {k}'

train = train_all[exact_train].reset_index(drop=True)
X_train, y_train = X_all[exact_train], train.pchembl.to_numpy()
if not np.isfinite(y_train).all(): raise ValueError('non-finite pChEMBL among exact-value training compounds')
external = external_all[~external_all.ic50_censored.astype(bool)].reset_index(drop=True)
X_ext, y_ext = fingerprint_matrix(external.canonical_smiles), external.pchembl.to_numpy()
y_cls_ext = external.active_ic50.to_numpy()
print(f'Regression training: {len(train):,} exact-value compounds (of {len(train_all):,}; {int((~exact_train).sum())} censored inactives excluded); {n_groups} scaffold groups')
print(f'External: {len(external):,} exact-value compounds (of {len(external_all):,}); {int(y_cls_ext.sum())} active at the classifier cutoff')
print('Fold validation sizes (exact only):', [len(va) for _, va in splits])

Regression training: 546 exact-value compounds (of 590; 44 censored inactives excluded); 283 scaffold groups
External: 36 exact-value compounds (of 36); 5 active at the classifier cutoff
Fold validation sizes (exact only): [84, 126, 58, 100, 178]


In [3]:
def reg_metrics(truth, pred):
    return {'rmse': mean_squared_error(truth, pred) ** .5, 'mae': mean_absolute_error(truth, pred),
            'r2': r2_score(truth, pred), 'spearman': spearmanr(truth, pred)[0], 'n': len(truth)}

models = {
    'mean_baseline': DummyRegressor(strategy='mean'),          # reference only; never selected
    'tanimoto_knn': TanimotoKNN(n_neighbors=5),
    'tanimoto_svr': SVR(kernel=tanimoto_kernel, C=10.0, epsilon=0.20),
    'random_forest': RandomForestRegressor(n_estimators=800, min_samples_leaf=2, max_features=0.50,
                                           n_jobs=CPU_WORKERS, random_state=RANDOM_STATE),
}
# Positions of exact-value compounds inside train_all -> map fold indices to the compact `train` table.
pos = np.full(len(train_all), -1); pos[exact_train] = np.arange(exact_train.sum())
rows, oof = [], {}
for name, base in models.items():
    pred = np.full(len(train), np.nan)
    for tr, va in splits:
        est = clone(base).fit(X_all[tr], train_all.pchembl.to_numpy()[tr])
        pred[pos[va]] = est.predict(X_all[va])
    assert np.isfinite(pred).all(), f'incomplete OOF predictions for {name}'
    oof[name] = pred
    fold_rmse = [mean_squared_error(train_all.pchembl.to_numpy()[va], pred[pos[va]]) ** .5 for _, va in splits]
    rows.append({'model': name, **reg_metrics(y_train, pred), 'rmse_fold_mean': np.mean(fold_rmse), 'rmse_fold_sd': np.std(fold_rmse, ddof=1)})
cv_metrics = pd.DataFrame(rows).sort_values('rmse').reset_index(drop=True)
display(cv_metrics.round(3))
cv_metrics.to_csv(OUT / 'regression_cv_metrics.csv', index=False)
pd.DataFrame({'canonical_smiles': train.canonical_smiles, 'pchembl': y_train, **{f'oof_{k}': v for k, v in oof.items()}}) \
  .to_csv(OUT / 'regression_cv_oof_predictions.csv', index=False)

# Selection uses training-only pooled out-of-fold RMSE; the baseline is excluded.
best_name = cv_metrics[cv_metrics.model != 'mean_baseline'].iloc[0].model
print('Selected from training-only CV:', best_name)

,model,rmse,mae,r2,spearman,n,rmse_fold_mean,rmse_fold_sd
0,tanimoto_svr,1.092,0.806,0.676,0.804,546,1.069,0.084
1,tanimoto_knn,1.170,0.838,0.628,0.768,546,1.142,0.093
2,random_forest,1.211,0.887,0.602,0.758,546,1.157,0.155
3,mean_baseline,1.931,1.596,-0.013,-0.140,546,1.936,0.021


Selected from training-only CV: tanimoto_svr


In [4]:
# Refit on all exact-value training data and evaluate on the external set ONCE (selected model only).
final_model = clone(models[best_name]).fit(X_train, y_train)
ext_pred = final_model.predict(X_ext)
ext = reg_metrics(y_ext, ext_pred)
ext['roc_auc_active_vs_inactive'] = roc_auc_score(y_cls_ext, ext_pred) if len(np.unique(y_cls_ext)) == 2 else np.nan
ext['n_active'] = int(y_cls_ext.sum())
ext_df = pd.DataFrame([{'model': best_name, **ext}])
display(ext_df.round(3))
ext_df.to_csv(OUT / 'regression_external_metrics.csv', index=False)
external.assign(predicted_pchembl=ext_pred).to_csv(OUT / 'regression_external_predictions.csv', index=False)

joblib.dump(final_model, MODEL_DIR / 'ace_pchembl_regressor.joblib')
(OUT / 'regressor_metadata.json').write_text(json.dumps({
    'training_database': 'CHEMBL1808', 'external_validation_database': 'CHEMBL2096989',
    'selected_model': best_name, 'selection_metric': 'pooled out-of-fold RMSE, StratifiedGroupKFold on Murcko scaffolds (folds shared with the classifier)',
    'assay_filter': 'biochemical/binding only', 'prodrugs_excluded': ace_config.EXCLUDE_PRODRUGS,
    'training_data': 'exact (=) IC50 values only',
    'n_splits': N_SPLITS, 'random_state': RANDOM_STATE, 'fingerprint': {'type': 'Morgan', 'radius': FP_RADIUS, 'n_bits': FP_SIZE},
    'training_compounds': len(train), 'external_compounds': len(external)}, indent=2), encoding='utf-8')
print(f'Locked regressor: {best_name}. External set is small ({len(external)} compounds, {int(y_cls_ext.sum())} actives) - interpret cautiously.')

,model,rmse,mae,r2,spearman,n,roc_auc_active_vs_inactive,n_active
0,tanimoto_svr,0.861,0.707,-0.328,0.407,36,0.661,5


Locked regressor: tanimoto_svr. External set is small (36 compounds, 5 actives) - interpret cautiously.


In [5]:
SOURCE_KEYS = ['FDA_DRUGBANK', 'EXPERIMENTAL_ZIAI', 'NPASS', 'PLANTS_CMAUP']
screened = []
for key in SOURCE_KEYS:
    path = INPUTS.get(key)
    if path is None or not Path(path).exists():
        print(f'Skipping missing library {key}'); continue
    library = ace_config.prepare_screening_library(Path(path), key)
    if library is None or library.empty:
        print(f'Skipping empty library {key}'); continue
    library = library.loc[library.canonical_smiles.notna()].copy()
    if 'source' not in library.columns: library['source'] = key
    if ace_config.EXCLUDE_PRODRUGS:
        library, prodrugs = ace_config.split_ace_prodrugs(library)
        prodrugs.to_csv(OUT / f'{key.lower()}_prodrugs_excluded.csv', index=False)
        print(f'{key}: excluded {len(prodrugs)} ACE-inhibitor prodrugs')
    X = fingerprint_matrix(library.canonical_smiles)
    library['predicted_pchembl'] = predict_in_chunks(final_model, X)
    library['predicted_ic50_nM'] = np.power(10.0, 9.0 - library.predicted_pchembl)
    library['max_train_tanimoto'] = max_tanimoto_similarity(X, X_train)
    library['model'] = best_name
    library.to_csv(OUT / f'{key.lower()}_pchembl_screen.csv', index=False)
    screened.append(library); print(f'{key}: screened {len(library):,} compounds')
if screened:
    all_screened = pd.concat(screened, ignore_index=True)
    all_screened.to_csv(OUT / 'all_libraries_pchembl_screen.csv', index=False)
    display(all_screened.sort_values('predicted_pchembl', ascending=False).head(20))
else:
    print('No libraries screened.')

FDA_DRUGBANK: excluded 13 ACE-inhibitor prodrugs
FDA_DRUGBANK: screened 2,438 compounds
EXPERIMENTAL_ZIAI: excluded 0 ACE-inhibitor prodrugs
EXPERIMENTAL_ZIAI: screened 7,445 compounds
NPASS: excluded 6 ACE-inhibitor prodrugs
NPASS: screened 91,920 compounds
PLANTS_CMAUP: excluded 1 ACE-inhibitor prodrugs
PLANTS_CMAUP: screened 57,855 compounds


,source,source_id,input_smiles,canonical_smiles,predicted_pchembl,predicted_ic50_nM,max_train_tanimoto,model
565,FDA_DRUGBANK,Lisinopril,NCCCCC(NC(CCc1ccccc1)C(=O)O)C(=O)N1CCCC1C(=O)O,NCCCCC(NC(CCc1ccccc1)C(=O)O)C(=O)N1CCCC1C(=O)O,8.599750,2.513331,1.000000,tanimoto_svr
1788,FDA_DRUGBANK,Enalaprilat,CC(NC(CCc1ccccc1)C(=O)O)C(=O)N1CCCC1C(=O)O,CC(NC(CCc1ccccc1)C(=O)O)C(=O)N1CCCC1C(=O)O,8.420110,3.800935,1.000000,tanimoto_svr
114585,PLANTS_CMAUP,IFGCUJZIWBUILZ-UHFFFAOYSA-N,CC(C)CC(NP(=O)(O)OC1OC(C)C(O)C(O)C1O)C(=O)NC(C...,CC(C)CC(NP(=O)(O)OC1OC(C)C(O)C(O)C1O)C(=O)NC(C...,8.099863,7.945788,1.000000,tanimoto_svr
22226,NPASS,NPC313640,CC(C)CC(NP(=O)(O)OC1OC(C)C(O)C(O)C1O)C(=O)NC(C...,CC(C)CC(NP(=O)(O)OC1OC(C)C(O)C(O)C1O)C(=O)NC(C...,8.099863,7.945788,1.000000,tanimoto_svr
63800,NPASS,NPC483990,CC(CS)C(=O)N1CCCC1C(=O)O,CC(CS)C(=O)N1CCCC1C(=O)O,7.489705,32.381328,1.000000,tanimoto_svr
1002,FDA_DRUGBANK,Captopril,CC(CS)C(=O)N1CCCC1C(=O)O,CC(CS)C(=O)N1CCCC1C(=O)O,7.489705,32.381328,1.000000,tanimoto_svr
144495,PLANTS_CMAUP,FAKRSMQSSFJEIM-RQJHMYQMSA-N,CC(CS)C(=O)N1CCCC1C(=O)O,CC(CS)C(=O)N1CCCC1C(=O)O,7.489705,32.381328,1.000000,tanimoto_svr
66109,NPASS,NPC479749,CC[C@H](C)C(NC(C)=O)C(=O)NC(Cc1ccc(O)cc1)C(=O)...,CC[C@H](C)C(NC(C)=O)C(=O)NC(Cc1ccc(O)cc1)C(=O)...,7.418786,38.125364,1.000000,tanimoto_svr
59017,NPASS,NPC479754,CC[C@H](C)C(NC(C)=O)C(=O)NC(CCC(=O)O)C(=O)N[C@...,CC[C@H](C)C(NC(C)=O)C(=O)NC(CCC(=O)O)C(=O)N[C@...,7.203110,62.645448,0.818182,tanimoto_svr
83913,NPASS,NPC479752,CC[C@H](C)C(NC(C)=O)C(=O)NC(CC(C)C)C(=O)N[C@@H...,CC[C@H](C)C(NC(C)=O)C(=O)NC(CC(C)C)C(=O)N[C@@H...,7.157048,69.654929,1.000000,tanimoto_svr
